# Data Cleaning

## 0. impoart libaries


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

dir_path = Path.cwd()

In [ ]:
# Helper function 
def inspect_data(dataframe):
    df = dataframe.copy()

    print("shape of df:", df.shape)
    print('-'*25)

    # null
    null_counts = df.isna().sum().rename("null_count").to_frame()
    print("Null value counts:")
    print(null_counts[null_counts['null_count'] > 0].to_string() if not null_counts[null_counts['null_count'] > 0].empty else "No null values found.")
    print('-'*25)

    # duplicates
    duplicate_count = df.duplicated().sum()
    print(f"Number of duplicate rows: {duplicate_count}")
    print('-'*25)

    # data types
    print("Data types of each column:")
    print(df.dtypes)
    print('-'*25)

    # summary statistics
    print("Summary statistics for numeric columns:")
    # print(df.describe().to_string())

    print(df.describe(include='number').T.to_string())
    print()
    print(df.describe(include='object').T.to_string())
    print('-'*25)

    # head
    print("First 5 rows of the dataframe:")
    print(df.head().to_string(index=False))

## 1. Load and explore the data

In [ ]:
data_dict = pd.read_csv(dir_path / 'data/maven_fuzzy_factory_data_dictionary.csv')

tables = data_dict['Table'].unique()

for table in tables:
    print(f"\nTable: {table}")
    table_data = data_dict[data_dict['Table'] == table]
    print(table_data[['Field' , 'Description']].to_string(index=False, justify='left', col_space=20))
    print("="*100)

In [ ]:
session = pd.read_csv(dir_path / 'data/website_sessions.csv')
display(session.head())

pageview = pd.read_csv(dir_path / 'data/website_pageviews.csv')
display(pageview.head())

In [ ]:
inspect_data(session)

- created_at type need to be converted to datetime
- null values must be checked and handled
- user_id and session_id must converted to object type

In [ ]:
inspect_data(pageview)

- ids must be converted to object type
- created_at must be converted to datetime type
  

## 2. Data cleaning 

### Sessions

In [ ]:
# Data types need to be fixed:
session['created_at'] = pd.to_datetime(session.created_at)
session['user_id'] = session.user_id.astype('object')
session['website_session_id'] = session['website_session_id'].astype('object')

In [ ]:
# null values
session_null_cases = (
    session.groupby(
        ['http_referer', 'utm_source', 'utm_campaign', 'utm_content'],
        dropna=False
    ).size().reset_index(name='count')
)

print(session_null_cases.to_string(justify='center', col_space=15))

Nothing need to be done for null values as they are not related to our analysis and they are not missing at random.

### Pageview

In [ ]:
# data type

pageview['created_at'] = pd.to_datetime(pageview.created_at)
pageview['website_session_id'] = pageview['website_session_id'].astype('object')
pageview['website_pageview_id'] = pageview['website_pageview_id'].astype('object')

## 3. Data Preparation 

In [ ]:
pageviews_agg =(
        pageview 
            .sort_values(['website_session_id', 'website_pageview_id']) 
            .groupby('website_session_id') \
                .agg(
                    first_pageview_url=('pageview_url', 'first'),
                    last_pageview_url=('pageview_url', 'last'),
                    concated_urls=('pageview_url', ' -> '.join),
                    pageview_count=('pageview_url', 'count'),
                    first_pageview_time = ('created_at', 'min'),
                    last_pageview_time = ('created_at', 'max')
                    ) 
        )

# adjustment for single pageview sessions: if the first and last pageview URLs are the same, we set the last_pageview_url to None
pageviews_agg.loc[
    pageviews_agg['first_pageview_url'] == pageviews_agg['last_pageview_url'],
    'last_pageview_url'
] = None

print('shape of pageviews_agg:', pageviews_agg.shape)
print("Aggregated pageviews data:")
pageviews_agg.head()

In [ ]:
session_pageviews = session.merge(pageviews_agg, on='website_session_id', how='left')

print('shape of session_pageviews:', session_pageviews.shape)
print("Merged session and pageviews data:")
session_pageviews.head()

## 4. Experiment Identification

as it's not clearly mentioned the exact scenario of experiment, we need to idenify it! 
-> finding the timeline of campaigns and the pages that are involved in the experiment

In [ ]:
# Landing page, ad content combination.
campaign_combinations = (
    session_pageviews
        .groupby(
            ['first_pageview_url', 'utm_content'],
            dropna=False
        ).size()
)

# Build a day-level timeline for each campaign combination.
campaign_timeline = (
    session_pageviews
        .groupby([session_pageviews['created_at'].dt.date, 'http_referer', 'utm_content', 'first_pageview_url'])
        .agg(row=('website_session_id', 'first')) # helper column to keep track of unique combinations for plotting
).reset_index()


# ===== Plotting the campaign timeline =====
# Assign each campaign combination a row number (1..N) for plotting.
temp_list = campaign_combinations.sort_index(level=0).index.values.tolist()

for i, landing in enumerate(temp_list):
    campaign_timeline.loc[
        (campaign_timeline['first_pageview_url'] == temp_list[i][0]) &
        (campaign_timeline['utm_content'] == temp_list[i][1]),
        'row'
    ] = i + 1

# Plot campaign activity over time.
plt.figure(figsize=(30, 6))

sns.scatterplot(
    data=campaign_timeline,
    x='created_at',
    y='row',
    hue='utm_content',
    alpha=0.7,
)

# Format plot: legend, labels, y-ticks, grid, title.
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylabel('campaign combinations (first pageview URL + utm_content)')
plt.yticks(range(1, len(temp_list) + 1), temp_list)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.title("""Campaign Timeline\n(first landing page based on the incoming traffic source)""")
plt.show()

A proper A/B test means:
- Same traffic source hitting two different pages
- Overlapping time window
- One starts after the other (the new challenger)

It's needed to identify the A/B test by looking for a lander that shares a traffic source with /home and has a clear start date where both are running simultaneously.

> `/home` vs `/lander-1` — sharing the `g_ad_1` traffic source. 
> The others (b_ad_2, g_ad_2, b_ad_1) never overlap on the same source, so they're not part of this A/B test.



In [ ]:
# Obtaining the A/B test date
print(
    session_pageviews[session_pageviews['first_pageview_url'].isin(['/home', '/lander-1'])] \
        .groupby(['first_pageview_url', 'utm_content']) \
        .agg(
            first_seen=('created_at', 'min'),
            last_seen=('created_at', 'max'),
            session_count=('website_session_id', 'count')
        ).to_string(justify='center', col_space=20)
)

In [ ]:
fig, ax = plt.subplots(figsize=(30, 6))

# Plot the campaign timeline
sns.scatterplot(
    data=campaign_timeline,
    x='created_at',
    y='row',
    hue='utm_content',
    alpha=0.7,
    ax=ax
)

# Define test window
test_start = pd.Timestamp('2012-06-19')
test_end = pd.Timestamp('2012-07-29')

# Highlight areas OUTSIDE the test window in gray
ax.axvspan(campaign_timeline['created_at'].min(), test_start, alpha=0.3, color='gray', label='Outside Test Window')
ax.axvspan(test_end, campaign_timeline['created_at'].max(), alpha=0.3, color='gray')

# Add vertical lines at test boundaries
ax.axvline(test_start, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax.axvline(test_end, color='green', linestyle='--', linewidth=2, alpha=0.7)

# Add text annotation
mid_point = test_start + (test_end - test_start) / 2
ax.text(mid_point, ax.get_ylim()[1] * 0.95, 'A/B Test Period\n(2012-06-19 → 2012-07-29)', 
        ha='center', fontsize=12, fontweight='bold', 
        bbox=dict(boxstyle='round,pad=0.8', facecolor='white', alpha=0.9))

ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylabel('campaign combinations (first pageview URL + utm_content)')
ax.set_yticks(range(1, len(temp_list) + 1))
ax.set_yticklabels(temp_list)
ax.set_xlabel('Date')
ax.set_title('Campaign Timeline with A/B Test Window Highlighted\n(first landing page based on the incoming traffic source)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.show()

**The overlap window runs from 2012-06-19 to 2012-07-29.**

## 5. Filtering the traffic for the A/B test Analysis 

In [ ]:
ab_test = session_pageviews[
    (session_pageviews['utm_content'] == 'g_ad_1') & # shared traffic source
    (session_pageviews['first_pageview_url'].isin(['/home', '/lander-1'])) & # landers in the test
    (session_pageviews['created_at'] >= '2012-06-19') & (session_pageviews['created_at'] <= '2012-07-29') # date range of the test
]

print(ab_test.groupby('first_pageview_url').agg(count=('website_session_id', 'size'))
      .assign(percentage=lambda x: (x['count'] / x['count'].sum() * 100).round(5))
        .to_string())

os.makedirs(dir_path / 'data/processed', exist_ok=True)
ab_test.to_parquet(dir_path / 'data/processed/session_pageviews_ab_test.parquet', index=False)
print(f"\nFiltered A/B test data saved to: { 'data/processed/session_pageviews_ab_test.parquet'}")